# 04 — Full Backtest

Run the complete funding rate arbitrage backtest and analyse results.

This notebook ties together:
1. Data loading from cache
2. Signal generation (z-score + basis momentum)
3. Simulation with Kalman hedge ratio
4. Performance analysis with QuantStats

In [ ]:
import sys

sys.path.insert(0, '..')

import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

from backtest.report import BacktestReport
from backtest.simulator import FundingRateSimulator, SimulatorConfig
from data.downloader import FundingRateDownloader
from research.funding_zscore import FundingZScore, ZScoreParams

In [ ]:
# Load config
with open('../config/default.yaml') as f:
    config = yaml.safe_load(f)
print(f"Config loaded: {len(config)} parameters")

In [ ]:
# Load data
funding = FundingRateDownloader.load_cached_data()
print(f"Funding records: {len(funding)}")
print(f"Symbols: {funding['symbol'].nunique()}")
print(f"Exchanges: {funding['exchange'].unique()}")
print(f"Date range: {funding['timestamp'].min()} → {funding['timestamp'].max()}")

In [ ]:
# Generate signals
zscore_params = ZScoreParams(
    lookback_periods=config.get('zscore_lookback', 90),
    entry_threshold=config.get('zscore_entry', 1.5),
    exit_threshold=config.get('zscore_exit', 0.0),
    min_annualised_rate=config.get('min_annualised_rate', 0.05),
)
zscore = FundingZScore(zscore_params)
signals = zscore.compute(funding)

entries = signals[signals['signal'] == 1]
exits = signals[signals['signal'] == -1]
print(f"Entry signals: {len(entries)}")
print(f"Exit signals: {len(exits)}")

In [ ]:
# Run simulation
sim_config = SimulatorConfig(
    initial_capital=config.get('initial_capital', 100_000),
    max_positions=config.get('max_positions', 10),
    max_position_pct=config.get('max_position_pct', 0.15),
    entry_zscore=config.get('zscore_entry', 1.5),
    exit_zscore=config.get('zscore_exit', 0.0),
    min_annualised_rate=config.get('min_annualised_rate', 0.05),
    use_kalman_hedge=config.get('use_kalman_hedge', True),
    exchange=config.get('primary_exchange', 'binance'),
)

simulator = FundingRateSimulator(sim_config)
state = simulator.run(funding, signals)

In [ ]:
# Generate report
report = BacktestReport(state)
metrics = report.metrics()

print("\n" + "="*50)
print("BACKTEST RESULTS")
print("="*50)
for k, v in metrics.items():
    print(f"  {k:25s}: {v}")

In [ ]:
# Equity curve
equity_df = state.to_equity_df()
if not equity_df.empty:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Equity Curve', 'Drawdown'])

    fig.add_trace(go.Scatter(x=equity_df.index, y=equity_df['equity'],
                             name='Equity', fill='tozeroy'), row=1, col=1)
    fig.add_trace(go.Scatter(x=equity_df.index, y=-equity_df['drawdown'],
                             name='Drawdown', fill='tozeroy',
                             line=dict(color='red')), row=2, col=1)

    fig.update_layout(title='Backtest Equity Curve & Drawdown', height=600)
    fig.show()

In [ ]:
# Trade analysis
trade_log = state.to_trade_log_df()
if not trade_log.empty:
    print(f"Total trades: {len(trade_log)}")
    print("\nTrade actions:")
    print(trade_log['action'].value_counts())

    closes = trade_log[trade_log['action'] == 'CLOSE']
    if 'net_pnl' in closes.columns:
        print(f"\nAvg P&L per trade: ${closes['net_pnl'].mean():.2f}")
        print(f"Win rate: {(closes['net_pnl'] > 0).mean():.1%}")

In [ ]:
# Save results
report.save('../results')
print("\nResults saved to ../results/")